# Feature Engineering on Primary Land Use Tax Lot Output (PLUTO) and High Volume For-Hire Vehicle (HVFHV) Demand Datasets:

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import geopandas as gpd
from shapely import wkt
import pandas as pd 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_pluto+demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"

PLUTO dataset:

In [ ]:
pluto_df_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_df_path)
pluto_df.head()

Zone dataset:

In [ ]:
zone_gdf_path = base_dir + '/developed/merged_data/zone_gdf.csv'
zone_gdf = pd.read_csv(zone_gdf_path)
zone_gdf['geometry'] = zone_gdf['geometry'].apply(wkt.loads)
zone_gdf.head()

Daily pickup demand:

In [ ]:
daily_pickup_demand_sdf_path = base_dir + '/developed/merged_data/daily_pickup_demand'
daily_pickup_demand_sdf = spark.read.parquet(daily_pickup_demand_sdf_path)
daily_pickup_demand_df = daily_pickup_demand_sdf.toPandas()
daily_pickup_demand_df.head()

Daily dropoff demand:

In [ ]:
daily_dropoff_demand_sdf_path = base_dir + '/developed/merged_data/daily_dropoff_demand'
daily_dropoff_demand_sdf = spark.read.parquet(daily_dropoff_demand_sdf_path)
daily_dropoff_demand_df = daily_dropoff_demand_sdf.toPandas()
daily_dropoff_demand_df.head()

# Find Daily Demand by Building Class:

In [ ]:
# Merge `pluto_df` and `daily_pickup_demand_df` on `location_id` and `PULocationID`
daily_pickup_demand_by_building_class_df = pd.merge(pluto_df, daily_pickup_demand_df, 
                                             left_on='location_id', 
                                             right_on='PULocationID', 
                                             how='left')

# Group by `building_class`, and calculate the sum of `daily_demand`
daily_pickup_demand_by_building_class_df = daily_pickup_demand_by_building_class_df.groupby(['building_class'])['daily_demand'] \
                                                                     .sum() \
                                                                     .reset_index()

# Sort by daily demand
daily_pickup_demand_by_building_class_df = daily_pickup_demand_by_building_class_df.sort_values(by='daily_demand', ascending=False)\
                                                                     .reset_index(drop=True)

daily_pickup_demand_by_building_class_df.head()

In [ ]:
daily_pickup_demand_by_building_class_df.head(24)

# Find Average Daily Demand by Location ID:

In [ ]:
zone_gdf = gpd.GeoDataFrame(zone_gdf, geometry='geometry')
zone_gdf = zone_gdf.drop_duplicates('location_id')

### Pickup:

In [ ]:
# Merge `zone_gdf` and `daily_demand_df` on `location_id` and `PULocationID`
daily_demand_by_location_df = pd.merge(zone_gdf, daily_pickup_demand_df, 
                                       left_on='location_id', 
                                       right_on='PULocationID', 
                                       how='left')

# Drop the 'PULocationID' column as it is no longer needed
daily_demand_by_location_df = daily_demand_by_location_df.drop('PULocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_demand`
daily_demand_by_location_df = daily_demand_by_location_df.groupby(['location_id'])['daily_demand'] \
                                                         .sum() \
                                                         .reset_index()

# Sort the DataFrame by `daily_demand` in descending order
daily_pickup_demand_by_location_df = daily_demand_by_location_df.sort_values(by='daily_demand', ascending=False) \
                                                         .reset_index(drop=True)

daily_pickup_demand_by_location_df.head()

### Dropoff:

In [ ]:
# Merge `zone_gdf` and `daily_dropoff_demand_df` on `location_id` and `DOLocationID`
daily_demand_by_location_df = pd.merge(zone_gdf, daily_dropoff_demand_df, 
                                       left_on='location_id', 
                                       right_on='DOLocationID', 
                                       how='left')

# Drop the 'DOLocationID' column as it is no longer needed
daily_demand_by_location_df = daily_demand_by_location_df.drop('DOLocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_demand`
daily_demand_by_location_df = daily_demand_by_location_df.groupby(['location_id'])['daily_demand'] \
                                                         .sum() \
                                                         .reset_index()

# Sort the DataFrame by `daily_demand` in descending order
daily_dropoff_demand_by_location_df = daily_demand_by_location_df.sort_values(by='daily_demand', ascending=False) \
                                                         .reset_index(drop=True)

daily_dropoff_demand_by_location_df.head()

# Save the Merged Dataset:

Daily demand by different building classes:

In [ ]:
demand_by_building_class_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_demand_by_building_class_df.csv'
demand_by_building_class_df_path = os.path.join(demand_by_building_class_df_dir, file_name)
daily_pickup_demand_by_building_class_df.to_csv(demand_by_building_class_df_path, index=False)

Daily pickup demand by different location ID:

In [ ]:
demand_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_pickup_demand_by_location.csv'
demand_by_location_df_path = os.path.join(demand_by_location_df_dir, file_name)
daily_pickup_demand_by_location_df.to_csv(demand_by_location_df_path, index=False)

Daily dropoff demand by different location ID:

In [ ]:
demand_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_dropoff_demand_by_location.csv'
demand_by_location_df_path = os.path.join(demand_by_location_df_dir, file_name)
daily_dropoff_demand_by_location_df.to_csv(demand_by_location_df_path, index=False)